In [5]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# ============================================================
# User settings
# ============================================================

ROOT = Path.cwd()

DATA_ROOT = ROOT / "Data" / "jacobian_data"
OUTPUT_ROOT = ROOT / "Results" / "jacobian_sensitivity"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

NODES = [4, 6, 8]

# Tick spacing per node
NODE_PLOT_CONFIGS = {
    4: {"x_tick_step": 10, "y_tick_step": 8},
    6: {"x_tick_step": 10, "y_tick_step": 16},
    8: {"x_tick_step": 20, "y_tick_step": 32},
}

# Font sizes
LABEL_FONTSIZE = 52
TICK_FONTSIZE = 34

DPI = 600

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 28,
    "font.weight": "bold",
    "axes.labelsize": LABEL_FONTSIZE,
    "axes.labelweight": "bold",
    "xtick.labelsize": TICK_FONTSIZE,
    "ytick.labelsize": TICK_FONTSIZE,
    "mathtext.fontset": "dejavuserif",
    "axes.linewidth": 1.8,
})


# ============================================================
# Helper functions
# ============================================================

def load_jacobian_matrices(results_dir: Path):
    """Load NN, PyTheus, and difference sensitivity matrices."""
    required_files = {
        "mean_nn": results_dir / "mean_nn_sensitivity.npy",
        "mean_py": results_dir / "mean_pytheus_sensitivity.npy",
        "difference": results_dir / "difference_sensitivity.npy",
    }

    for name, path in required_files.items():
        if not path.exists():
            raise FileNotFoundError(f"Missing {name} file: {path}")

    mean_nn = np.load(required_files["mean_nn"])
    mean_py = np.load(required_files["mean_py"])
    difference = np.load(required_files["difference"])

    return mean_nn, mean_py, difference


print("Current directory:", ROOT)
print("Data root:", DATA_ROOT)
print("Output root:", OUTPUT_ROOT)


Current directory: /home/bo48god/projects/deep_dreaming/PolySurrogate_temp/plotting_code
Data root: /home/bo48god/projects/deep_dreaming/PolySurrogate_temp/plotting_code/Data/jacobian_data
Output root: /home/bo48god/projects/deep_dreaming/PolySurrogate_temp/plotting_code/Results/jacobian_sensitivity


## Jacobian Plots for n=4,6 and 8


In [6]:
LABEL_FONTSIZE = 52
TICK_FONTSIZE = 34
CBAR_LABEL_FONTSIZE = 34
CBAR_TICK_FONTSIZE = 34
PANEL_FONTSIZE = 44

def style_axis_standalone(ax, matrix, x_tick_step, y_tick_step):
    x_ticks = np.arange(0, matrix.shape[1], x_tick_step)
    y_ticks = np.arange(0, matrix.shape[0], y_tick_step)

    ax.set_xticks(x_ticks)
    ax.set_yticks(y_ticks)
    ax.set_xticklabels(x_ticks, rotation=0)
    ax.set_yticklabels(y_ticks)

    ax.tick_params(axis="x", which="major", length=8, width=1.6, direction="out")
    ax.tick_params(axis="y", which="major", left=True, right=False, length=8, width=1.6, direction="out", labelleft=True)

    ax.set_ylabel("Output amplitude index", labelpad=18, fontsize=STANDALONE_LABEL_FONTSIZE, fontweight="bold")
    ax.set_xlabel("Input graph-weight index", labelpad=18, fontsize=STANDALONE_LABEL_FONTSIZE, fontweight="bold")
    ax.set_box_aspect(1)


STANDALONE_LABEL_FONTSIZE = 36

def plot_single_jacobian_panel(node, panel_name, matrix, cmap, vmin, vmax, cbar_label):
    """Create and save one self-contained heatmap (own colorbar, no panel label)."""
    output_dir = OUTPUT_ROOT / f"{node}_node"
    output_dir.mkdir(parents=True, exist_ok=True)

    x_tick_step = NODE_PLOT_CONFIGS[node]["x_tick_step"]
    y_tick_step = NODE_PLOT_CONFIGS[node]["y_tick_step"]

    fig, ax = plt.subplots(figsize=(14, 12))

    im = ax.imshow(
        matrix, origin="lower", interpolation="nearest",
        cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto",
    )

    style_axis_standalone(ax, matrix, x_tick_step, y_tick_step)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.06)
    cbar.set_label(cbar_label, fontsize=CBAR_LABEL_FONTSIZE, fontweight="bold", labelpad=20)
    cbar.ax.tick_params(labelsize=CBAR_TICK_FONTSIZE, width=1.6, length=7)
    plt.setp(cbar.ax.get_yticklabels(), fontweight="bold")

    fig.tight_layout()

    save_stem = output_dir / f"jacobian_sensitivity_{panel_name}_{node}_node"
    fig.savefig(save_stem.with_suffix(".png"), dpi=DPI, bbox_inches="tight")
    fig.savefig(save_stem.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(save_stem.with_suffix(".svg"), bbox_inches="tight")
    plt.close(fig)

    print("Saved:", save_stem.with_suffix(".png"))
    print("Saved:", save_stem.with_suffix(".pdf"))
    print("Saved:", save_stem.with_suffix(".svg"))


for node in NODES:
    print("\n" + "=" * 80)
    print(f"Node {node}")
    print("=" * 80)

    results_dir = DATA_ROOT / f"{node}_node"
    mean_nn, mean_py, difference = load_jacobian_matrices(results_dir)

    common_vmax = max(np.max(mean_nn), np.max(mean_py))

    plot_single_jacobian_panel(
        node, "nn_sensitivity", mean_nn, "viridis", 0.0, common_vmax,
        "Mean Jacobian sensitivity",
    )
    plot_single_jacobian_panel(
        node, "pytheus_sensitivity", mean_py, "viridis", 0.0, common_vmax,
        "Mean Jacobian sensitivity",
    )
    plot_single_jacobian_panel(
        node, "difference", difference, "magma", 0.0, np.max(difference),
        r"$|S^{\mathrm{NN}}_{ij} - S^{\mathrm{PM}}_{ij}|$",
    )

print("\nAll 9 separate jacobian figures generated (3 quantities x 3 nodes).")


Node 4
Saved: /home/bo48god/projects/deep_dreaming/PolySurrogate_temp/plotting_code/Results/jacobian_sensitivity/4_node/jacobian_sensitivity_nn_sensitivity_4_node.png
Saved: /home/bo48god/projects/deep_dreaming/PolySurrogate_temp/plotting_code/Results/jacobian_sensitivity/4_node/jacobian_sensitivity_nn_sensitivity_4_node.pdf
Saved: /home/bo48god/projects/deep_dreaming/PolySurrogate_temp/plotting_code/Results/jacobian_sensitivity/4_node/jacobian_sensitivity_nn_sensitivity_4_node.svg
Saved: /home/bo48god/projects/deep_dreaming/PolySurrogate_temp/plotting_code/Results/jacobian_sensitivity/4_node/jacobian_sensitivity_pytheus_sensitivity_4_node.png
Saved: /home/bo48god/projects/deep_dreaming/PolySurrogate_temp/plotting_code/Results/jacobian_sensitivity/4_node/jacobian_sensitivity_pytheus_sensitivity_4_node.pdf
Saved: /home/bo48god/projects/deep_dreaming/PolySurrogate_temp/plotting_code/Results/jacobian_sensitivity/4_node/jacobian_sensitivity_pytheus_sensitivity_4_node.svg
Saved: /home/bo48